# Social Media Brand Analytics: Netflix vs Amazon Prime Video

This notebook presents a cleaned portfolio version of an exploratory brand analytics workflow using public BlueSky posts. 
It covers data collection design, data cleaning, engagement analysis, sentiment analysis, topic modelling, commercial theme interpretation and micro-influencer screening.

**Portfolio note:** raw data and credentials are not included in this public repository. The notebook is structured so that it can be run when the relevant CSV snapshots are placed in the `data/` folder.

## 1. Imports and project configuration

In [ ]:
from pathlib import Path
import os
import re
import time
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from getpass import getpass
from collections import Counter

# Text analytics
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError:
    SentimentIntensityAnalyzer = None

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"

for path in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BRAND_KEYWORDS = {
    "Netflix": ["netflix", "#netflix"],
    "Amazon Prime Video": ["prime video", "amazon prime video"]
}

POSTS_PER_QUERY = 1000
RANDOM_STATE = 42

## 2. Optional BlueSky data collection

This section is provided for reproducibility. For public portfolio use, run the analysis from saved CSV snapshots rather than storing credentials or API tokens in the notebook.

In [ ]:
BASE_URL = "https://bsky.social/xrpc"

def create_bsky_session(handle: str, app_password: str) -> dict:
    """Create a BlueSky API session. Credentials should be entered at runtime, not stored in the notebook."""
    response = requests.post(
        f"{BASE_URL}/com.atproto.server.createSession",
        json={"identifier": handle, "password": app_password}
    )
    response.raise_for_status()
    return response.json()

def extract_post_record(post: dict) -> dict:
    """Extract common post fields from a BlueSky API response record."""
    author = post.get("author", {})
    record = post.get("record", {})
    return {
        "uri": post.get("uri"),
        "cid": post.get("cid"),
        "author_handle": author.get("handle"),
        "author_display_name": author.get("displayName"),
        "created_at": record.get("createdAt"),
        "text": record.get("text", ""),
        "like_count": post.get("likeCount", 0),
        "repost_count": post.get("repostCount", 0),
        "reply_count": post.get("replyCount", 0),
        "quote_count": post.get("quoteCount", 0),
        "language": ",".join(record.get("langs", [])) if isinstance(record.get("langs"), list) else None,
        "url": f"https://bsky.app/profile/{author.get('handle')}/post/{post.get('uri','').split('/')[-1]}"
    }

def search_bluesky_posts(session: dict, brand: str, keyword: str, max_posts: int = 1000, pause_seconds: float = 0.5) -> pd.DataFrame:
    """Collect posts for one brand-keyword pair using pagination."""
    headers = {"Authorization": f"Bearer {session['accessJwt']}"}
    rows, cursor = [], None

    while len(rows) < max_posts:
        params = {"q": keyword, "limit": min(100, max_posts - len(rows))}
        if cursor:
            params["cursor"] = cursor

        response = requests.get(
            f"{BASE_URL}/app.bsky.feed.searchPosts",
            headers=headers,
            params=params,
            timeout=30
        )
        response.raise_for_status()
        payload = response.json()

        for post in payload.get("posts", []):
            row = extract_post_record(post)
            row["brand"] = brand
            row["keyword_used"] = keyword
            rows.append(row)

        cursor = payload.get("cursor")
        if not cursor:
            break
        time.sleep(pause_seconds)

    return pd.DataFrame(rows)

# Example usage:
# handle = os.getenv("BSKY_HANDLE") or input("BlueSky handle: ")
# password = os.getenv("BSKY_APP_PASSWORD") or getpass("BlueSky app password: ")
# session = create_bsky_session(handle, password)
# all_posts = []
# for brand, keywords in BRAND_KEYWORDS.items():
#     for keyword in keywords:
#         all_posts.append(search_bluesky_posts(session, brand, keyword, POSTS_PER_QUERY))
# raw_posts = pd.concat(all_posts, ignore_index=True)
# raw_posts.to_csv(DATA_DIR / "raw_brand_posts_combined.csv", index=False)

## 3. Load saved snapshot

In [ ]:
raw_path = DATA_DIR / "raw_brand_posts_combined.csv"

if raw_path.exists():
    df_raw = pd.read_csv(raw_path)
else:
    df_raw = pd.DataFrame()
    print("Raw snapshot not found. Place raw_brand_posts_combined.csv in the data/ folder to run the full workflow.")

df_raw.shape

## 4. Cleaning and brand-analysis dataset

In [ ]:
def normalise_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def prepare_brand_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Create a deduplicated, brand-level dataset for analysis."""
    if df.empty:
        return df.copy()

    clean = df.copy()
    clean["text"] = clean["text"].fillna("").map(normalise_text)
    clean["created_at"] = pd.to_datetime(clean["created_at"], errors="coerce")
    clean["total_engagement"] = (
        clean[["like_count", "repost_count", "reply_count", "quote_count"]]
        .fillna(0)
        .sum(axis=1)
    )

    # Deduplicate using post URI where available, otherwise text-author combination.
    if "uri" in clean.columns:
        clean = clean.drop_duplicates(subset=["uri"])
    else:
        clean = clean.drop_duplicates(subset=["author_handle", "text", "created_at"])

    lower_text = clean["text"].str.lower()
    mentions_netflix = lower_text.str.contains(r"\bnetflix\b|#netflix", regex=True, na=False)
    mentions_prime = lower_text.str.contains(r"prime video|amazon prime video", regex=True, na=False)

    clean["mentions_netflix"] = mentions_netflix
    clean["mentions_prime_video"] = mentions_prime
    clean["cross_brand"] = mentions_netflix & mentions_prime

    # Keep posts assigned to one brand for comparative analysis.
    brand_analysis = clean.loc[~clean["cross_brand"]].copy()
    return brand_analysis

df_brand = prepare_brand_dataset(df_raw)
df_brand.to_csv(OUTPUT_DIR / "clean_brand_analysis.csv", index=False)
df_brand.head()

## 5. English-language subset for text modelling

In [ ]:
def is_english_language(value):
    if pd.isna(value):
        return False
    return "en" in str(value).lower().split(",")

if not df_brand.empty:
    df_text_en = df_brand.loc[df_brand["language"].apply(is_english_language)].copy()
    df_text_en = df_text_en.loc[df_text_en["text"].str.len() > 0].copy()
else:
    df_text_en = pd.DataFrame()

df_text_en.to_csv(OUTPUT_DIR / "clean_english_text_posts.csv", index=False)
df_text_en.shape

## 6. Descriptive and engagement analysis

In [ ]:
if not df_brand.empty:
    brand_summary = (
        df_brand.groupby("brand")
        .agg(
            clean_posts=("text", "size"),
            unique_users=("author_handle", "nunique"),
            avg_engagement=("total_engagement", "mean"),
            total_engagement=("total_engagement", "sum"),
            median_engagement=("total_engagement", "median"),
            max_engagement=("total_engagement", "max"),
        )
        .reset_index()
        .sort_values("clean_posts", ascending=False)
    )
    display(brand_summary)
    brand_summary.to_csv(OUTPUT_DIR / "clean_brand_summary.csv", index=False)
else:
    brand_summary = pd.DataFrame()
    print("No data loaded.")

In [ ]:
if not brand_summary.empty:
    ax = brand_summary.plot(
        x="brand",
        y=["clean_posts", "unique_users"],
        kind="bar",
        figsize=(8, 4)
    )
    ax.set_title("Brand visibility in cleaned dataset")
    ax.set_ylabel("Count")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "brand_visibility.png", dpi=150)
    plt.show()

In [ ]:
if not df_brand.empty:
    for brand_name, subset in df_brand.groupby("brand"):
        plt.figure(figsize=(7, 4))
        subset["engagement_capped"] = subset["total_engagement"].clip(upper=20)
        plt.hist(subset["engagement_capped"], bins=21)
        plt.title(f"Distribution of total engagement for {brand_name} (capped at 20)")
        plt.xlabel("Total engagement per post")
        plt.ylabel("Number of posts")
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"engagement_distribution_{brand_name.replace(' ', '_').lower()}.png", dpi=150)
        plt.show()

## 7. Hashtag and keyword analysis

In [ ]:
def extract_hashtags(text):
    return re.findall(r"#\w+", str(text).lower())

if not df_brand.empty:
    hashtag_rows = []
    for brand_name, subset in df_brand.groupby("brand"):
        hashtags = [tag for text in subset["text"] for tag in extract_hashtags(text)]
        hashtag_rows.append({
            "brand": brand_name,
            "posts_with_hashtags": subset["text"].apply(lambda x: len(extract_hashtags(x)) > 0).sum(),
            "total_hashtag_mentions": len(hashtags),
            "unique_hashtags": len(set(hashtags)),
            "top_hashtag": Counter(hashtags).most_common(1)[0][0] if hashtags else None
        })
    hashtag_summary = pd.DataFrame(hashtag_rows)
    display(hashtag_summary)
    hashtag_summary.to_csv(OUTPUT_DIR / "hashtag_summary.csv", index=False)
else:
    hashtag_summary = pd.DataFrame()

## 8. Sentiment analysis

In [ ]:
def classify_sentiment(compound):
    if compound >= 0.05:
        return "Positive"
    if compound <= -0.05:
        return "Negative"
    return "Neutral"

if SentimentIntensityAnalyzer is None:
    print("Install vaderSentiment to run sentiment scoring: pip install vaderSentiment")
elif not df_text_en.empty:
    analyzer = SentimentIntensityAnalyzer()
    df_text_en["sentiment_score"] = df_text_en["text"].apply(lambda x: analyzer.polarity_scores(str(x))["compound"])
    df_text_en["sentiment_label"] = df_text_en["sentiment_score"].apply(classify_sentiment)

    sentiment_summary = (
        pd.crosstab(df_text_en["brand"], df_text_en["sentiment_label"], normalize="index") * 100
    ).reset_index()

    for label in ["Positive", "Neutral", "Negative"]:
        if label not in sentiment_summary.columns:
            sentiment_summary[label] = 0

    sentiment_summary["net_sentiment"] = sentiment_summary["Positive"] - sentiment_summary["Negative"]
    display(sentiment_summary)
    sentiment_summary.to_csv(OUTPUT_DIR / "sentiment_summary_by_brand.csv", index=False)
else:
    sentiment_summary = pd.DataFrame()
    print("No English text data loaded.")

In [ ]:
if "sentiment_summary" in globals() and not sentiment_summary.empty:
    ax = sentiment_summary.set_index("brand")[["Positive", "Neutral", "Negative"]].plot(
        kind="bar",
        figsize=(8, 4)
    )
    ax.set_title("Sentiment distribution by brand")
    ax.set_ylabel("Percentage of English-language posts")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "sentiment_distribution_by_brand.png", dpi=150)
    plt.show()

## 9. Topic modelling with TF-IDF and NMF

In [ ]:
def clean_for_topic_model(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z#\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

if not df_text_en.empty:
    df_text_en["topic_text"] = df_text_en["text"].apply(clean_for_topic_model)

    stop_words = [
        "netflix", "prime", "video", "amazon", "watch", "watching", "show", "shows",
        "movie", "movies", "series", "season", "episode", "stream", "streaming"
    ]

    vectorizer = TfidfVectorizer(
        max_features=1000,
        min_df=3,
        max_df=0.80,
        stop_words="english"
    )
    X = vectorizer.fit_transform(df_text_en["topic_text"])
    nmf = NMF(n_components=5, random_state=RANDOM_STATE, init="nndsvda", max_iter=500)
    W = nmf.fit_transform(X)
    H = nmf.components_
    terms = np.array(vectorizer.get_feature_names_out())

    topic_words = []
    for topic_idx, weights in enumerate(H):
        top_terms = terms[np.argsort(weights)[::-1][:12]]
        topic_words.append({"topic": topic_idx, "top_terms": ", ".join(top_terms)})
    topic_table = pd.DataFrame(topic_words)
    display(topic_table)

    df_text_en["topic_id"] = W.argmax(axis=1)
    topic_table.to_csv(OUTPUT_DIR / "nmf_topic_terms.csv", index=False)
else:
    topic_table = pd.DataFrame()
    print("No English text data loaded.")

## 10. Commercial theme mapping

The original coursework manually interpreted topic and keyword outputs into commercial themes. In a production workflow, this mapping should be reviewed by an analyst and validated on post samples.

In [ ]:
THEME_KEYWORDS = {
    "Content releases and trailers": ["trailer", "release", "new", "season", "episode", "series", "film", "movie"],
    "Viewer opinion and recommendation": ["recommend", "review", "love", "best", "good", "bad", "watching"],
    "Platform availability and access": ["available", "rent", "buy", "platform", "youtube", "apple", "access"],
    "Live sport and event streaming": ["nba", "wnba", "sport", "match", "game", "live"],
    "Pricing, subscriptions and ads": ["price", "subscription", "ads", "ad", "cost", "pay"],
    "App, playback and technical experience": ["app", "playback", "buffer", "quality", "error", "technical"]
}

def assign_themes(text):
    text_l = str(text).lower()
    found = []
    for theme, keywords in THEME_KEYWORDS.items():
        if any(keyword in text_l for keyword in keywords):
            found.append(theme)
    return found

if not df_text_en.empty:
    theme_rows = []
    for _, row in df_text_en.iterrows():
        themes = assign_themes(row["topic_text"])
        for theme in themes:
            theme_rows.append({"brand": row["brand"], "theme": theme})
    theme_df = pd.DataFrame(theme_rows)

    if not theme_df.empty:
        theme_summary = (
            theme_df.groupby(["brand", "theme"]).size()
            .reset_index(name="posts")
            .merge(df_text_en.groupby("brand").size().reset_index(name="brand_posts"), on="brand")
        )
        theme_summary["pct_of_brand_posts"] = 100 * theme_summary["posts"] / theme_summary["brand_posts"]
        display(theme_summary.sort_values(["brand", "pct_of_brand_posts"], ascending=[True, False]))
        theme_summary.to_csv(OUTPUT_DIR / "commercial_theme_summary.csv", index=False)
else:
    theme_summary = pd.DataFrame()

## 11. Micro-influencer candidate scoring

In [ ]:
def is_likely_publisher(handle):
    handle = str(handle).lower()
    return handle.endswith(".com") or "news" in handle or "media" in handle or handle == "handle.invalid"

if not df_brand.empty:
    creator_summary = (
        df_brand.groupby(["brand", "author_handle"])
        .agg(
            posts_about_brand=("text", "size"),
            total_engagement=("total_engagement", "sum"),
            average_engagement=("total_engagement", "mean"),
            median_engagement=("total_engagement", "median"),
            max_engagement=("total_engagement", "max")
        )
        .reset_index()
    )

    creator_summary["publisher_like_account"] = creator_summary["author_handle"].apply(is_likely_publisher)
    eligible = creator_summary.loc[~creator_summary["publisher_like_account"]].copy()

    # Normalise within brand to avoid one metric dominating.
    score_cols = ["posts_about_brand", "total_engagement", "average_engagement", "median_engagement"]
    for col in score_cols:
        eligible[col + "_rank"] = eligible.groupby("brand")[col].rank(pct=True)

    eligible["micro_influencer_score"] = (
        0.25 * eligible["posts_about_brand_rank"] +
        0.25 * eligible["total_engagement_rank"] +
        0.25 * eligible["average_engagement_rank"] +
        0.25 * eligible["median_engagement_rank"]
    )

    recommendations = (
        eligible.sort_values(["brand", "micro_influencer_score"], ascending=[True, False])
        .groupby("brand")
        .head(1)
        .reset_index(drop=True)
    )
    display(recommendations[["brand", "author_handle", "posts_about_brand", "total_engagement", "average_engagement", "median_engagement", "micro_influencer_score"]])
    recommendations.to_csv(OUTPUT_DIR / "micro_influencer_recommendations.csv", index=False)
else:
    recommendations = pd.DataFrame()
    print("No brand data loaded.")

## 12. Portfolio conclusions

The analysis should be read as a public social media snapshot. The strongest strategic conclusion is that Netflix had broader visibility, while Amazon Prime Video had stronger engagement intensity and a more favourable sentiment profile in the analysed English-language posts. Future work should extend the collection window, add release-event controls and enrich creator screening with follower/network quality metrics.